# Prefix caching and quantization

*Local AI Inference Lab: Serving, Optimization and Application Quality*

**Question:** How do we make local AI faster and more efficient without losing useful task performance?

**Learn:** separate prefill/decode and test FP16/AWQ × caching with paired cold/warm requests.
**Requirements:** [CPU replay prerequisites](../SETUP.md); [Live configuration](../tutorial/RUN_ALL.md) only for fresh GPU runs.
**Expected outputs:** recorded tables, a server command/request/evaluated SQL example,
and an observation → explanation → limitation → decision for every experiment.

The recorded Qwen FP16 cache-on run has cold/warm median TTFT of **2.019 → 0.083 s**, while request E2E changes **6.038 → 4.039 s**. Prefill reuse does not eliminate generation time. New to the workload? Read [00 — local serving](00_local_serving.ipynb), or start here for optimization.

These are observations from a small recorded Dell workload, not universal claims. A faster
response is not necessarily a better answer; different optimizations affect different stages.
Clinical-trial analytics supplies an executable application-quality test. We have not evaluated
multiple application domains. The synthetic database contains no patient data.


## 1. Choose how to explore

| Mode | Behavior |
|---|---|
| **Replay — default** | Load checksum-verified **recorded Dell measurements** and reproduce tables without a GPU. |
| Plan | Show configuration and server commands without launching or downloading anything. |
| Live | Launch owned servers, benchmark, preserve evidence and clean up. Requires Linux GPU setup. |

Choose **Run All** now for replay. To change mode, create `tutorial/config/vllm-qwen.local.json`
using [the shared setup guide](../tutorial/RUN_ALL.md) and set `mode` to `replay`, `plan` or `live`.
All machine settings and flags belong in this configuration file. A live failure never silently
loads archived results. Replay ignores new experiment settings when presenting historical values.

Serving target: **Qwen3-4B**, `Qwen/Qwen3-4B`. The checkpoint is loaded as **FP16** through
native vLLM in a separate GPU Python environment.
The AWQ W4A16 candidate uses 4-bit weights and FP16 activations. The KV cache is not quantized by this weight format.

Four distinct clinical questions are repeated five times per serving condition. They are not
20 independent clinical tasks. Application quality separately evaluates four cases, including
planning, validation, SQL execution and one repair. Exact-SQL semantic review is still pending.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.workflow import Study
from clinical_trial_mvp.tutorial.teaching import server_example, request_example, response_example, interpret
if 'study' in globals():
    study.close()
study = Study('vllm', 'qwen', root=ROOT)
results = study.results
print('Mode:', study.mode)
print('Model:', 'Qwen/Qwen3-4B')
print(study.prepare())


Mode: replay
Model: Qwen/Qwen3-4B
Replay: recorded Dell measurements; no model download, server, GPU or network calls.


## 2. Predict before measuring

Prefill processes the prompt; decode generates output tokens. A cache can reuse shared
prefix blocks. Quantization changes weight representation and kernels. Shorter context
changes both input work and the information available. More concurrency can improve
aggregate throughput while increasing per-request latency.

The application checks plan structure, SQL safety, schema use, execution and reference-result
match. Endpoint HTTP success alone says nothing about whether a useful answer was delivered.
Before each experiment, predict both its systems effect and its possible quality cost.


| | Cache off | Cache on |
|---|---|---|
| FP16 | Baseline | Prefix reuse |
| AWQ W4A16 | Weight representation | Combined effects |

Hold schema, four questions, seed 42, temperature 0, cap 1024 and context limit 8192 fixed.
An AWQ SQL compatibility smoke precedes all four live variants. For cache-on, each target
uses **reset → cold target → reset → different-question prime → identical target warm**,
with the identical shared schema prefix. Resets must acknowledge `success=true`.
Startup, health checks, warmup, reset, prime and SQL scoring are outside request timing.

The new native controller sets `VLLM_SERVER_DEV_MODE=1` consistently to expose reset controls.
The original reset-404 failure remains part of the evidence. See [its case study](../tutorial/CASE_STUDIES.md).


## 3. Compatibility before benchmarking

Smoke asks whether the path can generate and execute SQL. It is not a general quality score.
Replay loads the recorded smoke when available. Plan makes no model calls. Live stops on
failed required smoke, retains the failure and never substitutes replay measurements.


In [2]:
smoke = study.smoke()
if isinstance(smoke, dict):
    print(smoke.get('_recorded', {}).get('label', study.mode), smoke.get('status'), smoke.get('reason', ''))
else:
    print(smoke)


recorded Dell measurement completed 


## 4. Experiments and decisions

Read each prediction before running its cell. Quality counts accompany speed; unavailable metrics stay unavailable.


### What does an FP16 request cost without prefix caching?


In [3]:
result = study.run('fp16_cache_off')
print(result.get('_recorded', {}).get('label', study.mode), 'fp16_cache_off:', result['status'])
display(Markdown(interpret(study, 'fp16_cache_off')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'fp16_cache_off')))


recorded Dell measurement fp16_cache_off: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T221400_918883Z.json](../experiment-tracker/mvp4/runs/20261106T221400_918883Z.json)

**Prediction:** Establish executable SQL and separate application quality from request speed.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 6.081/7.653 s; aggregate output: 23.952 tok/s.

**Explanation:** Transport timing and application correctness measure different stages.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use this baseline to judge useful completed tasks, not latency alone.

### Under the hood: three inspectable paths

**A. Server command.** The next cell shows the recorded portable argv or the planned command,
model revision, dtype, cache control and scheduler limits. A setting absent from the archive
is marked unknown rather than inferred.
[Native launcher](../src/clinical_trial_mvp/serving/launch.py).


In [4]:
display(JSON(server_example(study, 'fp16_cache_off')))


<IPython.core.display.JSON object>

**B. One HTTP request.** Inspect the full prompt, generation options and schema
contract. This uses the exact payload builder called by the transport. Replay reconstructs
it from the saved prompt and recorded flags; it is not a packet capture. No request is sent
by this inspection cell. [Request builder and timing](../src/clinical_trial_mvp/serving/client.py).


In [5]:
display(JSON(request_example(study, 'fp16_cache_off')))


<IPython.core.display.JSON object>

**C. One evaluated response.** Inspect generated SQL, safety/schema validation,
execution outcome, query rows, the reference query and the recorded result comparison.
A reference match is not a human semantic review. [Plan/SQL validation](../src/clinical_trial_mvp/agent/analyst.py)
· [reference scorer](../src/clinical_trial_mvp/evaluation/scorers.py).


In [6]:
display(JSON(response_example(study, 'fp16_cache_off')))


<IPython.core.display.JSON object>

### Can shared-prefix reuse reduce first-token latency?


In [7]:
result = study.run('fp16_cache_on')
print(result.get('_recorded', {}).get('label', study.mode), 'fp16_cache_on:', result['status'])
display(Markdown(interpret(study, 'fp16_cache_on')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'fp16_cache_on')))


recorded Dell measurement fp16_cache_on: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T224700_600112Z.json](../experiment-tracker/mvp4/runs/20261106T224700_600112Z.json)

**Prediction:** Shared-prefix reuse should reduce warm prefill latency more than total generation time.

**Observation:** Cold → warm TTFT p50: 2.019 → 0.083 s. Request E2E p50: 6.038 → 4.039 s. Direct SQL result match: 10/20 cold; 10/20 warm.

**Explanation:** The paired phase records are consistent with avoiding repeated prefill; generation still consumes time.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Consider caching for repeated schema/system prefixes after checking both phase quality counts.

### How does W4A16 change latency and task results?


In [8]:
result = study.run('awq_w4a16_cache_off')
print(result.get('_recorded', {}).get('label', study.mode), 'awq_w4a16_cache_off:', result['status'])
display(Markdown(interpret(study, 'awq_w4a16_cache_off')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'awq_w4a16_cache_off')))


recorded Dell measurement awq_w4a16_cache_off: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T230608_780309Z.json](../experiment-tracker/mvp4/runs/20261106T230608_780309Z.json)

**Prediction:** Four-bit weights reduce weight storage; latency and SQL quality may change in either direction.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 4.310/5.123 s; aggregate output: 34.040 tok/s.

**Explanation:** Quantization changes weight representation and kernel behavior; caching addresses prefix reuse.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Benchmark compatible AWQ when weight capacity matters; no native VRAM reduction is established here.

### Do caching and W4A16 help together?


In [9]:
result = study.run('awq_w4a16_cache_on')
print(result.get('_recorded', {}).get('label', study.mode), 'awq_w4a16_cache_on:', result['status'])
display(Markdown(interpret(study, 'awq_w4a16_cache_on')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'awq_w4a16_cache_on')))


recorded Dell measurement awq_w4a16_cache_on: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/runs/20261106T232408_684730Z.json](../experiment-tracker/mvp4/runs/20261106T232408_684730Z.json)

**Prediction:** Shared-prefix reuse should reduce warm prefill latency more than total generation time.

**Observation:** Cold → warm TTFT p50: 2.083 → 0.057 s. Request E2E p50: 4.271 → 2.258 s. Direct SQL result match: 15/20 cold; 15/20 warm.

**Explanation:** The paired phase records are consistent with avoiding repeated prefill; generation still consumes time.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Consider caching for repeated schema/system prefixes after checking both phase quality counts.

## 5. Put performance beside quality

TTFT is first generated content. Request E2E measures generation/transport; application E2E
includes planning, validation, execution and repair. Cold/warm cache phases are separate from
supplemental warmed summaries. Semantic review remains unreviewed. Native VRAM telemetry
failed and stays missing; no memory-reduction conclusion is supported by those runs.


In [10]:
display(Markdown(study.report()))
display(Markdown(study.cache_report()))


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/Optimization/fp16_cache_off · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 2.05 | 6.08/7.65 | 23.95 | — |
| MVP4/Optimization/fp16_cache_on · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 0.08 | 4.06/5.70 | 34.60 | — |
| MVP4/Optimization/awq_w4a16_cache_off · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 2.10 | 4.31/5.12 | 34.04 | — |
| MVP4/Optimization/awq_w4a16_cache_on · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2344 | 0.06 | 2.28/3.09 | 62.67 | — |

| Experiment | Application E2E p50/p95 s | Request TTFT p95 s |
|---|---|---|
| fp16_cache_off · recorded Dell measurement | 12.851/15.079 | 2.052 |
| fp16_cache_on · recorded Dell measurement | 9.004/10.428 | 0.100 |
| awq_w4a16_cache_off · recorded Dell measurement | 9.684/12.191 | 2.113 |
| awq_w4a16_cache_on · recorded Dell measurement | 5.925/7.814 | 0.076 |

| Variant | Phase | SQL match | Semantic | TTFT p50/p95 s | E2E p50/p95 s | TPOT p50 s | tok/s | Peak VRAM GiB |
|---|---|---|---|---|---|---|---|---|
| fp16_cache_off · recorded Dell measurement | cache_off | 10/20 | unreviewed | 2.005/2.014 | 5.994/7.601 | 0.028 | 24.214 | — |
| fp16_cache_on · recorded Dell measurement | cold | 10/20 | unreviewed | 2.019/2.033 | 6.038/7.644 | 0.028 | 24.074 | — |
| fp16_cache_on · recorded Dell measurement | warm | 10/20 | unreviewed | 0.083/0.102 | 4.039/5.691 | 0.028 | 34.677 | — |
| awq_w4a16_cache_off · recorded Dell measurement | cache_off | 15/20 | unreviewed | 2.070/2.078 | 4.241/5.076 | 0.015 | 34.463 | — |
| awq_w4a16_cache_on · recorded Dell measurement | cold | 15/20 | unreviewed | 2.083/2.096 | 4.271/5.110 | 0.016 | 34.238 | — |
| awq_w4a16_cache_on · recorded Dell measurement | warm | 15/20 | unreviewed | 0.057/0.077 | 2.258/3.079 | 0.016 | 62.896 | — |

### Check output work before explaining AWQ latency

Compare output-token counts by phase and case before attributing E2E differences entirely
to faster computation. The next table uses the same target records as the paired cache
results. Its decode rate is a client estimate, not GPU kernel timing. Generated SQL/text
can differ even at similar token lengths.


In [11]:
from clinical_trial_mvp.tutorial.token_analysis import analyze, table
token_analysis = analyze(study.results)
display(Markdown(table(token_analysis)))


| Configuration | Phase | Requests | Mean output tokens | Median output tokens | Estimated decode tok/s p50 |
|---|---|---|---|---|---|
| FP16 | cold | 20 | 152.00 | 143.00 | 35.14 |
| FP16 | warm | 20 | 150.75 | 140.50 | 35.17 |
| AWQ W4A16 | cold | 20 | 151.50 | 143.00 | 64.24 |
| AWQ W4A16 | warm | 20 | 151.50 | 143.00 | 64.17 |

### Filled interpretation of the historical Qwen comparison

**Observation:** with caching enabled, FP16 cold/warm TTFT was **2.019 → 0.083 s** and
E2E was **6.038 → 4.039 s**; AWQ was **2.083 → 0.057 s** and **4.271 → 2.258 s**.
Direct reference matches stayed **10/20** in each FP16 phase and **15/20** in each AWQ phase.
**Explanation:** prefix reuse is consistent with less prefill work; AWQ also changed the
generation path and actual SQL. **Limitation:** this small historical workload does not
establish general speed/quality gains, reviewed semantics or native memory savings.
**Decision:** test caching for repeated schemas; adopt a validated AWQ checkpoint only after
checking useful task results and latency on the target machine. These conclusions interpret
the archive; any fresh Live run must be assessed using its own tables.

In the recorded runs, mean output lengths are FP16/AWQ **152/151.5 tokens cold**
and **150.75/151.5 warm**. Shorter output alone is not a sufficient aggregate explanation.
Similar lengths do not establish identical token content or isolate GPU compute.
[Per-case token audit and interpretation](../tutorial/RESULTS.md#output-length-before-awq-attribution).
Fresh live counts appear in the table above; these quoted values remain historical.


### Recorded evidence figure

![Recorded Dell measurement](../tutorial/figures/01_prefix_cache.png)

This figure always describes the archived Dell runs, including when the notebook is in live
mode. Fresh measurements appear in the tables above. [Figure sources](../tutorial/figures/figure_data.json)
· [reproduction script](../scripts/tutorial/render_findings.py).


## 6. Finish, preserve and continue

Save with Ctrl+S. Replay changes only the notebook display; it does not rewrite measurements.
Live adds unique records and owns only the processes it starts.
Native servers stop after each row; the final cleanup cell is safe to repeat.
Interrupted/failed operations clean up owned servers; a kernel/OS crash requires explicit
cleanup using the saved process group.


In [12]:
print(study.close())


Study servers stopped. Saved records and notebook outputs are retained.


Continue to [the next chapter](02_nemotron_optimization.ipynb), or read:

- [What should I choose?](../tutorial/RESULTS.md#what-should-i-choose) — concise results and deployment decisions.
- [Troubleshooting case studies](../tutorial/CASE_STUDIES.md).
- [Four-case SQL review packet](../tutorial/SQL_REVIEW.md) — human review pending.
- [Fresh GPU validation checklist](../tutorial/GPU_VALIDATION.md).
- [Original executed notebook](history/dell-2026-11-07/local%20AI%20optimization%20-%20Qwen.ipynb) and [checksum manifest](history/dell-2026-11-07/MANIFEST.json).

Historical iteration labels and source lineage are preserved in the evidence, not used as
the learner-facing chapter structure.
